In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from gensim.models import Word2Vec

import pandas as pd
import numpy as np

data = pd.read_csv("../data/text_preprocessed.csv")

In [2]:
data.sample(1, random_state=0)

,Unnamed: 0.1,Unnamed: 0,ProductId,UserId,HelpfulnessNumerator,HelpfulnessDenominator,Score,Summary,Text,review_date,helpfulness_score,review_length,total_reviews,Text_processed
47725,47725,44718,B001EQ55RW,A3GJGBF7FO8WDJ,0,1,3,"A good idea, but too bitter for my taste","The nuts come in a handy package, and they sme...",2008-08-21,0.0,65,123,nut come handy package smell good taste dry bi...


In [3]:
tfidf = TfidfVectorizer(
    max_features=5000,      
    min_df=5,               
    max_df=0.8,             
    ngram_range=(1, 2)      
)


Esse trecho cria um **vetorizador TF-IDF**, responsável por transformar textos em vetores numéricos que representam a **importância das palavras** (ou combinações de palavras) dentro do conjunto de documentos.

A ideia do TF-IDF é:

**destacar termos frequentes em um documento mas raros no conjunto total**

**Assim, palavras realmente informativas ganham mais peso, enquanto termos muito comuns perdem relevância.**

Esse vetorizador será usado para:

**identificar palavras-chave mais relevantes**

**gerar métricas interpretáveis**

**alimentar análises posteriores (ex: agregação por produto)**

In [4]:
X_tfidf = tfidf.fit_transform(data["Text_processed"])

Essa etapa converte os textos preprocessados em vetores TF-IDF, transformando linguagem natural em uma representação numérica que permite identificar termos relevantes de forma estatística.

In [5]:
feature_names = np.array(tfidf.get_feature_names_out())

def top_tfidf_terms_by_product(df, X, top_n=10):
    results = {}

    for product_id, idxs in df.groupby("ProductId").indices.items():
        mean_tfidf = X[idxs].mean(axis=0)
        mean_tfidf = np.asarray(mean_tfidf).ravel()

        top_idx = mean_tfidf.argsort()[-top_n:][::-1]
        results[product_id] = feature_names[top_idx].tolist()

    return results


Esta função agrega os vetores TF-IDF por produto e extrai os termos mais relevantes, permitindo identificar palavras-chave que representam o conteúdo textual das avaliações de cada produto.

### Exemplo do produto B001EQ55RW : 

In [6]:
product_tfidf_terms = top_tfidf_terms_by_product(
    data, X_tfidf, top_n=10
)

print(product_tfidf_terms["B001EQ55RW"])


['almond', 'cocoa', 'chocolate', 'nut', 'emerald', 'snack', 'almonds', 'taste', 'dark chocolate', 'dark']


Estas são as palavras mais relevantes para o produto em questão.

### Tokenizando :

In [7]:
sentences = data["Text_processed"].str.split()

### WOrd2vec :

In [8]:
w2v_model = Word2Vec(
    sentences=sentences,
    vector_size=100,
    window=5,
    min_count=5,
    workers=4,
    sg=1  
)


O Word2Vec transforma palavras em vetores numéricos que capturam significado e contexto.
Palavras que aparecem em contextos parecidos ficam com vetores próximos no espaço vetorial
(ex.: “good” perto de “great”, longe de “bad”).

### Exemplo do nosso produto em questão :

In [9]:
w2v_model.wv.most_similar("chocolate", topn=5)

[('cocoa', 0.7785897254943848),
 ('fudge', 0.7734806537628174),
 ('Ghiradelli', 0.746479868888855),
 ('chocolatey', 0.7307226061820984),
 ('choco', 0.7254495620727539)]